## **CitiBike Jersey City - Urban Mobility Analysis**

### **About the Project**

The project analyses the different use cases for citibikes in Jersey City area and tries to find a way to optimize the citibike infrastructure to cope with the commuter demand.

### **About the Dataset**

This dataset contains the different details of the citibike commuter history across Jersey City area like ride date and time, ride start and end points, etc.

The dataset contains 13 columns:-
1. ride_id: This is the unique identifier for rides across the dataset.
2. rideable_type: This shows the type of bike used for a particular ride.
3. started_at: Shows the start time and date of the ride.
4. ended_at: Shows the end time and date of the ride.
5. start_station_name: Shows the station where the ride started.
6. start_station_id: The start station id.
7. end_station_name: Shows the station where the ride ended.
8. end_station_id: The end station id.
9. start_lat: The latitude for the start station coordinates.
10. start_lng: The longitude for the start station coordinates.
11. end_lat: The latitude for the end station coordinates.
12. end_lng: The longitude for the end station coordinates.
13. member_casual: Shows if the rider is a member or not.

Analysis is limited to May 2026 data and may not reflect seasonal usage patterns.

### **About this Notebook**

This notebook dives into the exploratory data analysis of the May 2026 citibike data for Jersey City area. Different types of checks are performed on the dataset like null check, duplicate check. data consistency check, data validation and column datatype check.

The cleaned data is finally saved as a new dataset file into the project folder.

### **Importing Necessary Libraries**

In [16]:
import pandas as pd

### **Loading Dataset**

In [17]:
df = pd.read_csv("../data/JC-202605-citibike-tripdata.csv")

### **Exploratory Data Analysis**

**Checking Dataset Head**

In [18]:
df.head(5)

,"q""ride_id""",rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,82D531671F07CEE6,electric_bike,2026-05-29 09:21:48.430,2026-05-29 09:46:51.394,14 St Ferry - 14 St & Shipyard Ln,HB202,E 53 St & Lexington Ave,6617.09,40.752961,-74.024353,40.758281,-73.970694,member
1,BC3888F5C97704D9,classic_bike,2026-05-30 15:39:35.546,2026-05-30 15:42:41.986,Paulus Hook,JC002,Exchange Pl,JC116,40.714145,-74.033552,40.716366,-74.034344,member
2,D0AF14E7B533B3D0,classic_bike,2026-05-25 22:13:29.172,2026-05-25 22:16:12.461,Washington St & Morgan St,JC149,Exchange Pl,JC116,40.719446,-74.036892,40.716366,-74.034344,member
3,AB6C74D6382A3361,classic_bike,2026-05-07 10:06:14.421,2026-05-07 10:13:30.958,Newport PATH,JC066,Exchange Pl,JC116,40.727224,-74.033759,40.716366,-74.034344,member
4,663D67EA2B3FBEA0,electric_bike,2026-05-23 19:52:24.311,2026-05-23 20:22:34.932,Bergen Ave,JC095,Van Nostrand Ave & Ocean Ave,JC132,40.722104,-74.071455,40.699830,-74.084160,casual


**Checking Dataset Shape and Column Datatypes**

In [19]:
#Checking the shape of the dataset
print(df.shape)

#Checking the datatypes of all columns in the dataset
print(df.dtypes)

(95350, 13)
q"ride_id"                str
rideable_type             str
started_at                str
ended_at                  str
start_station_name        str
start_station_id          str
end_station_name          str
end_station_id            str
start_lat             float64
start_lng             float64
end_lat               float64
end_lng               float64
member_casual             str
dtype: object


**Changing the Datatype of the needed columns to appropriate datatypes**

In [20]:
#Adding the needed columns to the list
cols = ['started_at', 'ended_at']

#Traversing the list and changing the datatypes of the columns
for col in cols:
    df[col] = pd.to_datetime(df[col])

**Rechecking the column datatypes to confirm**

In [21]:
df.dtypes

q"ride_id"                       str
rideable_type                    str
started_at            datetime64[us]
ended_at              datetime64[us]
start_station_name               str
start_station_id                 str
end_station_name                 str
end_station_id                   str
start_lat                    float64
start_lng                    float64
end_lat                      float64
end_lng                      float64
member_casual                    str
dtype: object

**Checking the null values in the dataset**

In [22]:
df.isnull().sum()

q"ride_id"              0
rideable_type           0
started_at              0
ended_at                0
start_station_name      0
start_station_id        0
end_station_name      293
end_station_id        355
start_lat               0
start_lng               0
end_lat               355
end_lng               355
member_casual           0
dtype: int64

The null check above revealed 355 missing values across the end station columns (end_station_name, end_station_id, end_lat, end_lng).

**Dropping the rows having null values in the columns**

Dropping these **355 records** instead of imputing these values because these values amount to only **0.4%** of the total records, this is not a significant enough amount to influence the rest of the data.


In [23]:
df_cleaned = df.dropna(subset = ['end_station_name','end_station_id','end_lat','end_lng'])

**Updated Shape of the dataset**

In [24]:
df_cleaned.shape

(94995, 13)

**Checking duplicates**

In [25]:
df_cleaned.duplicated().sum()

np.int64(0)

**Checking if there are any rows with reversed timestamps,i.e, having start times ahead of end times.**

In [26]:
(df_cleaned['started_at'] > df_cleaned['ended_at']).sum()

np.int64(0)

**Checking if there are any values or rows that have coordinates outside of the New Jersey area.**

In [27]:
((df_cleaned['start_lat'] < 40.68) | (df_cleaned['start_lat'] > 40.77) | 
 (df_cleaned['end_lat'] < 40.68) | (df_cleaned['end_lat'] > 40.77) |
 (df_cleaned['start_lng'] < -74.10) | (df_cleaned['start_lng'] > -74.02) | 
 (df_cleaned['end_lng'] < -74.10) | (df_cleaned['end_lng'] > -74.02)).sum()

np.int64(306)

The dataset contains both Jersey City (JC prefix) and Hoboken (HB prefix) stations, both in New Jersey. The coordinate bounds above cover the full NJ service area. The 306 flagged rows have end coordinates falling outside NJ in Manhattan and show numeric decimal end station IDs corresponding to NYC Citi Bike docks. These are removed as operational anomalies in the next step.

**Removing Cross-System Anomalies.**

306 rides show numeric decimal end station IDs (e.g. 5105.01, 6617.09) corresponding to NYC Citi Bike docks. All start stations are NJ-based (JC or HB prefix) but these rides end at NYC stations. These are physically impossible as customer trips and are likely operational data artifacts. They represent 0.32% of the dataset and are removed.

In [28]:
df_cleaned = df_cleaned[
    df_cleaned['end_station_id'].str.startswith(('JC','HB'), na=False)
]
print(df_cleaned.shape)

(94689, 13)


**Checking for any irregular data in the member_casual column.**

In [29]:
df_cleaned['member_casual'].unique()

<StringArray>
['member', 'casual']
Length: 2, dtype: str

Only expected values found: 'member' and 'casual'

**Checking for any irregular data in the rideable_type column.**

In [30]:
df_cleaned['rideable_type'].unique()

<StringArray>
['classic_bike', 'electric_bike']
Length: 2, dtype: str

Only expected values found: 'electric_bike' and 'classic_bike

**Updating the name of the identifying column**

In [31]:
df_cleaned.columns
df_cleaned = df_cleaned.rename(columns={'q"ride_id"':'ride_id'})

**Verifying if the name has been updated in the dataset**

In [32]:
df_cleaned.columns

Index(['ride_id', 'rideable_type', 'started_at', 'ended_at',
       'start_station_name', 'start_station_id', 'end_station_name',
       'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng',
       'member_casual'],
      dtype='str')

As we can see from the above result. The identifying column name has been updated.

**Exporting the cleaned csv file to the project folder**

In [33]:
df_cleaned.to_csv('../data/JC-202605-citibike-tripdata-cleaned.csv', index=False)